In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup

In [2]:
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS =  10//2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]

In [3]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [10]:
df = pd.read_csv(train_path)
df['rule']= df['rule'].str.lower().str.strip()
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

no advertising: spam, referral links, unsolicited advertising, and promotional content are not allowed.


In [11]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [12]:
test_df = pd.read_csv(test_path)
test_df['rule']= test_df.rule.str.lower().str.strip()

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.unique())}
augmented_df['rule_id']= augmented_df.rule.map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"no advertising: spam, referral links, unsolici...",0.0,"no advertising: spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"no advertising: spam, referral links, unsolici...",0.0,"no advertising: spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [14]:
unlabelled= pd.read_csv('/kaggle/input/jigsaw-unlabelled-14b/sampled_unlabelled_100k_with_predictions.csv')
unlabelled['rule']= unlabelled['rule'].str.lower().str.strip()
unlabelled['text']= unlabelled['rule']+ ' [SEP] '+ unlabelled['body']
unlabelled['rule_id']= unlabelled.rule.map(rule_map)

In [15]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df, 
            test_size=0.2, 
            stratify=augmented_df["rule"], 
            random_state=seed
        )
        unlabelled['label']= unlabelled.rule_violation.round(0)
        #change to keep the actual test set out of llm predictions.
        unlabelled= unlabelled.query('text not in @augmented_df.text.values')

        confident_positives = unlabelled[unlabelled['rule_violation'] >= 0.85]

        # Take confident negatives (pred <= 0.1)
        confident_negatives = unlabelled[unlabelled['rule_violation'] <= 0.1]
        
        # Sample same number of negatives + 10%
        n_positives = len(confident_positives)
        n_negatives_to_sample = int(n_positives * 1.1)
        
        sampled_negatives = confident_negatives.sample(
          n=min(n_negatives_to_sample, len(confident_negatives)),
          random_state=seed
        )
        
        # Combine
        unlabelled_taken = pd.concat([confident_positives, sampled_negatives], ignore_index=True)

        unlabelled_taken= unlabelled_taken.sample(frac=.8, random_state=seed)# take 80% of all the points for data diversity
        print(f'Seed {seed} - Pseudo: {len(confident_positives)} pos (>=0.85), {len(sampled_negatives)} neg (<=0.1), total={len(unlabelled_taken)}')

        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        unlabelled_taken.to_csv(f'fixed_pseudo_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}, pseudo={len(unlabelled_taken)}')

Seed 42 - Pseudo: 10734 pos (>=0.85), 11807 neg (<=0.1), total=18033
Seed 42 splits saved: train=1500, val=375, pseudo=18033
Seed 123 - Pseudo: 10734 pos (>=0.85), 11807 neg (<=0.1), total=18033
Seed 123 splits saved: train=1500, val=375, pseudo=18033


In [16]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len,weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids
        self.weights = weights if weights is not None else [1.0]*len(texts)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        item['weights'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

In [17]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [18]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, mask)
        weights = batch["weights"].to(device)
        loss = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
        loss = (loss * weights).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [19]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [20]:
def train_model_seed(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed}] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    unlabelled_taken = pd.read_csv(f'fixed_pseudo_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist()+unlabelled_taken['text'].tolist(), 
        train_data['label'].tolist()+unlabelled_taken['label'].tolist(), 
        train_data['rule_id'].tolist()+unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN,
        [1.0]*len(train_data) + [0.3]*len(unlabelled_taken)

    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed}] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed}] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_seed_{seed}.bin")
    
    print(f"[Seed {seed}] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    mp.set_start_method('fork', force=True)
    
    processes = []
    for idx, seed in enumerate(SEEDS):
       gpu_id = idx % torch.cuda.device_count()
       p = mp.Process(target=train_model_seed, args=(seed, gpu_id))
       p.start()
       processes.append(p)
    
    for p in processes:
       p.join()

    import json
    results = []
    for seed in SEEDS:
      with open(f'results_seed_{seed}.json', 'r') as f:
          results.append(json.load(f))
    
    aucs = [r['best_auc'] for r in results]
    losses = [r['best_loss'] for r in results]
    
    print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"Loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

    print("All models trained!")

[Seed 42] Training on cuda:0
[Seed 123] Training on cuda:1


2025-10-06 22:07:43.957424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-06 22:07:43.957551: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759788463.980080     848 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759788463.980079     851 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759788463.987165     848 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1759788463.987166     851 cuda_blas.cc:1

[Seed 42] Epoch 1/5


Training:   0%|          | 0/611 [00:00<?, ?it/s]

[Seed 123] Epoch 1/5


...

Training:   5%|▌         | 31/611 [00:26<08:17,  1.17it/s]

[Seed 123] Loss: 0.1201, Val Loss: 0.6272, Val AUC: 0.8766


Training:   5%|▌         | 32/611 [00:27<08:15,  1.17it/s]

[Seed 123] Epoch 2/5


Training:  73%|███████▎  | 446/611 [06:48<02:31,  1.09it/s]

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
    all_preds = []
    
    for seed in SEEDS:
        device = torch.device("cuda:0")
        model = JigsawModel(MODEL_PATH).to(device)
        model.load_state_dict(torch.load(f"model_seed_{seed}.bin", map_location=device))
        model.eval()
        
        test_preds = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Inference seed {seed}"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask)
                test_preds.extend(torch.sigmoid(logits).cpu().numpy())
        
        all_preds.append(test_preds)
    
    ensemble_preds = np.mean(all_preds, axis=0)
    
    sample = pd.read_csv(sample_sub_path)
    sample["rule_violation"] = ensemble_preds
    sample.to_csv("submission.csv", index=False)
    print(f"Ensembled {len(SEEDS)} models")
else:
    !touch submission.csv
    
!head -n 4 submission.csv

In [ ]:
#changes: 2 seeds (42, 123) with different splits, trained in parallel on 2 GPUs, ensembled predictions
#ideas to test-> check if psedo labels matter-yep they help, at what multiplier, at what weight,at what class balance, at what confidence, extra -ve random sampling

In [ ]:
#ablation
#start                            .8765 3rd epoch .4731 val loss
#all pseudo 
#random sample pseudo 2k          .8871 3nd epoch, .4731 val loss
#random sample pseudo 4k          .8977 .8761 2.5th epoch, .4133 .4663 val loss
#random sample pseudo 8k          .8979 .8932 3.5th epochs, .4223 .4409
#random sample pseudo 16k         .9082 .8904 3th epoch  .4356 .4363 
#random sample pseudo 32k         .8893 .9104 2.5th epoch  .4494 .4286 ***
#random sample pseudo 64k         .8913 3th epoch  .4298
#random sample pseudo 100k        .8913 3th epoch  .4298

# weightitng starts 
#.3 weights

# good options
#. 5 weight, hard labels, remove .1-.85, balance to have same, 20k points taken, dontoverwhelm with pseudo data, for each rule 10k points, so at best have 20k points(10k pairs), at best we have 10k points, so use them with diff wights .5-

In [ ]:
#TODO keep the 2 rules trained with pseudo data and train daat present, then for other rules keep it just train data as no pseudo data for those rules is present, and infer with those seperate models for the 2 psedo data rules, for others just infere with the typical model trained woithout any pseudo data.